In [11]:
# =============================================================================
# TEAM HAUSANLP - SEMEVAL 2026 TASK 9 (SUBTASK 2)
# Official Paper Implementation: Afro-XLMR-Large + 1:3 Undersampling + Weighted BCE
# =============================================================================

import pandas as pd
import numpy as np
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import Sampler, DataLoader
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from google.colab import files

# --- 1. CONFIGURATION (Matching Paper Specs) ---
MODEL_NAME = "Davlan/afro-xlmr-large" # Using the Large variant
LABELS = ['political', 'racial/ethnic', 'religious', 'gender/sexual', 'other']
MAX_LEN = 128
BATCH_SIZE = 4
GRAD_ACCUM = 4
EPOCHS = 6
LEARNING_RATE = 2e-5

# --- 2. DATA LOADING ---
print(" Loading Data...")
train_df = pd.read_csv('hau_dev.csv') # Ensure your training file is named exactly this
test_df = pd.read_csv('hau_trai.csv') # Ensure your test file is named exactly this

# --- 3. WEIGHTED LOSS CALCULATION (Inverse Frequency) ---
print("\n Calculating Class Weights (Inverse Frequency)...")
pos_counts = train_df[LABELS].sum().values
total_samples = len(train_df)

# Formula from paper: (Total - Positive) / Positive
inverse_freq_weights = (total_samples - pos_counts) / pos_counts
class_weights = torch.tensor(inverse_freq_weights, dtype=torch.float).to("cuda" if torch.cuda.is_available() else "cpu")

print("Calculated Weights (wc):")
for i, label in enumerate(LABELS):
    print(f" - {label.upper()}: {inverse_freq_weights[i]:.1f}")

# --- 4. PREPROCESSING & TOKENIZATION ---
print("\n Tokenizing (Max Length: 128)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)

def format_labels(batch):
    return {'labels': [[float(batch[l][i]) for l in LABELS] for i in range(len(batch['id']))]}

train_ds = train_ds.map(format_labels, batched=True)
test_ds = test_ds.map(format_labels, batched=True)

#  THE FIX: Remove string columns so PyTorch can build numeric tensors
cols_to_remove = ['id', 'text']
train_ds = train_ds.remove_columns([c for c in cols_to_remove if c in train_ds.column_names])
test_ds = test_ds.remove_columns([c for c in cols_to_remove if c in test_ds.column_names])
print(" String columns removed for PyTorch compatibility.")

# --- 5. DYNAMIC UNDERSAMPLING (1:3 RATIO BATCH SAMPLER) ---
print("\n Setting up Dynamic Undersampling (1:3 Ratio)...")

# Identify polarized vs non-polarized indices
train_labels = np.array([train_ds[i]['labels'] for i in range(len(train_ds))])
is_polarized = train_labels.sum(axis=1) > 0

pos_indices = np.where(is_polarized)[0].tolist()
neg_indices = np.where(~is_polarized)[0].tolist()

class OneToThreeBatchSampler(Sampler):
    """Custom sampler that strictly yields 1 polarized and 3 non-polarized samples per batch."""
    def __init__(self, pos_idx, neg_idx, batch_size=4):
        self.pos_idx = pos_idx
        self.neg_idx = neg_idx
        self.batch_size = batch_size
        self.num_batches = len(self.pos_idx) # 1 epoch = seeing all positive samples once

    def __iter__(self):
        pos_copy = list(self.pos_idx)
        neg_copy = list(self.neg_idx)
        random.shuffle(pos_copy)
        random.shuffle(neg_copy)

        for i in range(self.num_batches):
            batch = [pos_copy[i]] # 1 Polarized
            for j in range(3):    # 3 Non-polarized
                neg_pointer = (i * 3 + j) % len(neg_copy)
                batch.append(neg_copy[neg_pointer])
            random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches

# --- 6. CUSTOM TRAINER ARCHITECTURE ---
print("\n Initializing Afro-XLMR-Large Model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    problem_type="multi_label_classification"
)

class SemEvalTrainer(Trainer):
    # Override Dataloader to inject our 1:3 Sampler
    def get_train_dataloader(self):
        train_dataset = self.train_dataset
        data_collator = self.data_collator
        sampler = OneToThreeBatchSampler(pos_indices, neg_indices, batch_size=BATCH_SIZE)
        return DataLoader(
            train_dataset,
            batch_sampler=sampler,
            collate_fn=data_collator,
            drop_last=False,
            num_workers=0
        )

    # Override Loss Function to use Weighted BCE safely inside AMP
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # THE FIX: Dynamically match the datatypes of weights and labels to the logits.
        # This keeps the PyTorch AMP (Mixed Precision) graph completely intact.
        target_dtype = logits.dtype
        weights = class_weights.to(device=logits.device, dtype=target_dtype)
        labels = labels.to(dtype=target_dtype)

        loss_fct = nn.BCEWithLogitsLoss(pos_weight=weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# We use fp16=True to fit the Large model on Colab's T4 GPU
args = TrainingArguments(
    output_dir="./results_official",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    fp16=False, # Crucial for 'Large' model memory management
    save_strategy="no",
    report_to="none"
)

trainer = SemEvalTrainer(
    model=model, args=args, train_dataset=train_ds, data_collator=data_collator
)

# --- 7. TRAINING & PREDICTION ---
print("\n STARTING OFFICIAL TRAINING...")
trainer.train()

print("\n Generating Predictions on Test Set...")
test_output = trainer.predict(test_ds)
test_probs = 1 / (1 + np.exp(-test_output.predictions))

# Standard 0.5 threshold
final_preds = (test_probs >= 0.50).astype(int)

# --- 8. SAVING SUBMISSION ---
submission = pd.DataFrame(final_preds, columns=LABELS)
submission.insert(0, 'id', test_df['id'].values)
filename = "submission_hau_PAPER_OFFICIAL.csv"
submission.to_csv(filename, index=False)

print(f"\n SUCCESS! File '{filename}' is ready.")
files.download(filename)


 Loading Data...

 Calculating Class Weights (Inverse Frequency)...
Calculated Weights (wc):
 - POLITICAL: 3.3
 - RACIAL/ETHNIC: 4.4
 - RELIGIOUS: 1.4
 - GENDER/SEXUAL: 6.1
 - OTHER: 2.4

 Tokenizing (Max Length: 128)...


Map:   0%|          | 0/185 [00:00<?, ? examples/s]

Map:   0%|          | 0/286 [00:00<?, ? examples/s]

Map:   0%|          | 0/185 [00:00<?, ? examples/s]

Map:   0%|          | 0/286 [00:00<?, ? examples/s]

...

Step,Training Loss



 Generating Predictions on Test Set...



 SUCCESS! File 'submission_hau_PAPER_OFFICIAL.csv' is ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>